# Homework 9: Classification, Outliers, and Hypothesis Testing

This homework covers three lectures:
* **Lecture 26** (classification: k-nearest neighbors) -- Problem 1
* **Lecture 27** (identifying outliers) -- Problem 2
* **Lecture 28** (hypothesis testing) -- Problem 3

Problem 4 is a capstone: ceramic fracture strength follows Weibull, not normal, statistics --
you'll run the exact same two-sample workflow from Problem 3 on it, plus one more idea
(spread, quantified by the Weibull modulus, not just the mean).

Four datasets, all familiar:
* `dataset_steels` -- 915 steel alloys (Lecture 26's `alloy_family` classification target)
* `dataset_alloy_cleaning` -- messy elemental property records (Lecture 27's outlier dataset)
* `dataset_3dprinting` -- 50 FDM prints (Lecture 28's two-sample dataset)
* `dataset_weibull_ceramic_strength` -- synthetic ceramic fracture strengths (new today)

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

*Only* the `ipynb` file type will be accepted.

Save a copy of the notebook right away to avoid losing your work!

# Problem 0 (0 pts): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work -- critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

## Data file for Problem 1

## Dataset: Steel Alloy Compositions and Mechanical Properties

This dataset contains elemental compositions and mechanical properties of 915 steel alloys.

Composition columns (in wt%): `C`, `Si`, `Mn`, `P`, `S`, `Ni`, `Cr`, `Mo`, `Cu`, `V`, `Al`, `N`, `Ceq`, `Nb + Ta`

Target columns: `0.2% Proof Stress (MPa)`, `Tensile Strength (MPa)`, `Elongation (%)`, `Reduction in Area (%)`

The `Alloy code` column contains labels like `A1`, `B3`, etc. We derive `alloy_family` (first letter only)
to get a small set of categorical labels useful for classification tasks (4 families: C, L, M, V, all >150 samples).

In [1]:
import os
import pandas as pd

_file = 'steels.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

# derive alloy family label (first letter of alloy code)
data['Alloy family'] = [c[0] for c in data['Alloy code']]

# set up features (composition columns) and target
x = data.loc[:, ' C':'Nb + Ta']
y = data['Alloy family']
alloy_family = data['Alloy family']

print(f'{len(data)} samples, {x.shape[1]} composition features')
data.head()

915 samples, 14 composition features


,Alloy code,C,Si,Mn,P,S,Ni,Cr,Mo,Cu,...,Al,N,Ceq,Nb + Ta,Temperature (°C),0.2% Proof Stress (MPa),Tensile Strength (MPa),Elongation (%),Reduction in Area (%),Alloy family
0,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,27,342,490,30,71,M
1,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,100,338,454,27,72,M
2,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,200,337,465,23,69,M
3,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,300,346,495,21,70,M
4,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,400,316,489,26,79,M


# Problem 1 (25 pts): kNN with a fresh feature pair

Lecture 26 classified alloy family from **Cr and Ni** content and got strong accuracy (97-98%
even with just those two features). Today you'll try a pair the lecture never touched, and a
wider range of `k`, to see whether every feature pair works that well.

Reuse `knn_predict` exactly as written in Lecture 26 (copy it into the cell below -- it is not
re-included here on purpose, so you have to bring the tool with you, same as any function you'd
reuse from a prior lecture):

```python
def knn_predict(xtrain, ytrain, xquery, k):
    "Classify each row of xquery by majority vote among its k nearest neighbors in xtrain."
    xtrain = np.asarray(xtrain)
    ytrain = np.asarray(ytrain)
    preds = []
    for q in np.asarray(xquery):
        distances = np.sqrt(np.sum((xtrain - q) ** 2, axis=1))
        nearest = np.argsort(distances)[:k]
        labels, counts = np.unique(ytrain[nearest], return_counts=True)
        preds.append(labels[np.argmax(counts)])
    return np.array(preds)
```

(a) Redefine `x = data[[' S', ' Al']]` -- sulfur and aluminum content, a pair Lecture 26 never
used. This replaces the include's default `x` (all 14 composition columns), exactly like
Lecture 20 redefining `x` before its own train/test split. `y` is still the default
`alloy_family`; do not redefine it.

(b) Run the include cell below to get `idx_train`/`idx_test`/`xtrain`/`xtest`/`ytrain`/`ytest`,
built from whatever `x` part (a) just set -- so `xtrain`/`xtest` below are the S/Al columns.

## Train/Test Split

In order to evaluate the performance of a model on unseen data, we need
to *hold out* some data. This will be called the **test** data.
The data used to fit will be called the **training** data.

As we use more sophisticated models, it is vitally important that we
evaluate our models in this way to avoid *overfitting*.
Remember that a model with enough degrees of freedom can perfectly fit
any data, but it won't have any predictive power!

```
all data:    [########################################################]
shuffle, then slice into two pieces:
train (75%): [##########################################]
test  (25%):                                              [##########]
```

We shuffle the *indices* once, then slice both `x` and `y` with that same shuffled order --
so a row's features and its target always stay paired.

In [2]:
import numpy as np

# split indices rather than data -- more flexible
rng = np.random.default_rng(0)
shuffled = rng.permutation(data.index)
n_test = int(np.ceil(0.25 * len(shuffled)))
idx_test = shuffled[:n_test]
idx_train = shuffled[n_test:]

xtrain = x.loc[idx_train]
xtest  = x.loc[idx_test]

ytrain = y.loc[idx_train]
ytest  = y.loc[idx_test]

print(f"Train: {xtrain.shape[0]} samples, Test: {xtest.shape[0]} samples")

Train: 686 samples, Test: 229 samples


(c) For `k in [1, 10, 25, 50, 90]`, compute and print train accuracy and test accuracy using
the `xtrain`/`xtest` from part (b). (Accuracy = fraction of predictions matching the true
label -- same definition Lecture 26 used, no `eval_helpers` needed; that module is for
R²/RMSE, a different kind of score.)

(d) Now repeat part (c) with **all 14 composition columns** instead of just S and Al:
`x_all = data.loc[:, ' C':'Nb + Ta']`, then `xtrain_all = x_all.loc[idx_train]` and
`xtest_all = x_all.loc[idx_test]` -- reuse the *same* `idx_train`/`idx_test` from part (b)
(don't re-run the include; the split only depends on `data.index`, not on `x`'s columns, so
the row split is identical either way). Print the same train/test accuracy table for the same
five `k` values.

(e) In 3-4 sentences: how does S/Al's accuracy compare to Lecture 26's Cr/Ni pair, at small
`k`? Metallurgically, sulfur and aluminum are minor players (impurity control, deoxidizing) --
does that help explain why they separate alloy family worse than chromium and nickel do? Does
adding the other 12 composition columns in part (d) close the gap, and at every `k` you tried
or only some?

## Data file for Problem 2

## Dataset: Metal Alloy Elemental Properties (uncleaned)

In [3]:
import os
import pandas as pd

_file = 'alloy_cleaning.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github, encoding='latin1')
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e
data.head()

,number,name,symbol,name_symbol,pronunciation,appearance,atomic_number,group_block,period,element_category,...,band_gap,recognised_as_an_element_by,curie_point,recognized_as_a_unique_metal_by,recognized_as_a_distinct_element_by,thermal_diffusivity,tensile_strength,molar_volume,proposed_formal_name,alternative_names
0,1,Hydrogen,H,"hydrogen, H","/?ha?dr?d??n/, HY-dr?-j?n",colorless gas,1,"group 1, s-block",1,diatomic nonmetal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Helium,He,"helium, He","/?hi?li?m/, HEE-lee-?m","colorless gas, exhibiting a red-orange glow wh...",2,"group 18 (noble gases), s-block",1,noble gas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Lithium,Li,"lithium, Li","/?l??i?m/, LI-thee-?m",silvery-white,3,"group 1 (alkali metals), s-block",2,alkali metal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Beryllium,Be,"beryllium, Be","/b??r?li?m/, b?-RIL-ee-?m",white-gray metallic,4,"group 2 (alkaline earth metals), s-block",2,alkaline earth metal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Boron,B,"boron, B",/?b??r?n/,black-brown,5,"group 13, p-block",2,metalloid,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Problem 2 (25 pts): outliers on a fresh pair of columns

Lecture 27 ran the z-score and IQR rules on `melting_point`, `thermal_conductivity`, and (in
the Check-Your-Understanding) `electrical_resistivity`. Today: two different columns,
`youngs_modulus` and `thermal_expansion`. Same two rules, same coercion step first (both
columns are stored as text with unit strings and qualifiers mixed in).

(a) Coerce `youngs_modulus` to numeric with `pd.to_numeric(data['youngs_modulus'],
errors='coerce')`. Compute the z-score of every valid value. Flag anything with `abs(z) > 3`.
Print how many elements are flagged and their symbols, z-scores, and raw values.

(b) Coerce `thermal_expansion` the same way. Compute Q1, Q3, and the IQR fence
(`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`). Print the fence bounds, how many elements fall outside it,
and their symbols and values.

(c) Look at the rows where `youngs_modulus` is present in the raw data but *fails* to parse
(`data['youngs_modulus'].notna() & ym.isna()`, using whatever variable name you gave the
coerced column in part (a)). Print those rows' `symbol` and raw `youngs_modulus` text. Pick
**one** and explain in one sentence what specifically breaks the parse (a units qualifier, a
concatenated range, an allotrope prefix, etc.).

(d) Pick the element flagged in part (a) and one element flagged in part (b). For each, write
2-3 sentences: is it a **legitimate extreme value** (real physics, keep it), a **parsing/data-
entry artifact** (fixable or droppable), or a **genuine ambiguous case** (segment or caveat,
don't just delete)? Use Lecture 27's three categories by name, and justify your call using the
element's actual chemistry or the raw text you saw in part (c) -- not just "it's far from the
mean."

## Data file for Problem 3

## Dataset: 3D Printing (FDM) Parameters

Data come from [this github repository](https://github.com/ahmetokudan/3dprinterdeeplearning)
and are provided with an unlimited license (public domain).

In [4]:
import os
import pandas as pd

_file = '3dprinting.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e
data.head()  # show a view of the data file

,layer_height (mm),wall_thickness (mm),infill_density (%),infill_pattern,nozzle_temperature (degC),bed_temperature (degC),print_velocity (mm/s),material,fan_speed (%),roughness (microns),tension_strength (Mpa),elongation (%)
0,0.02,8,90,grid,220,60,40,abs,0,25,18,1.2
1,0.02,7,90,honeycomb,225,65,40,abs,25,32,16,1.4
2,0.02,1,80,grid,230,70,40,abs,50,40,8,0.8
3,0.02,4,70,honeycomb,240,75,40,abs,75,68,10,0.5
4,0.02,6,90,grid,250,80,40,abs,100,92,5,0.7


# Problem 3 (25 pts): two-sample tests on a fresh comparison

Lecture 28 compared `infill_pattern` and `material` against `tension_strength (Mpa)`, plus
`material` against `elongation (%)` in the Check-Your-Understanding. Today: two comparisons
the lecture never ran, using the same four-step workflow (state H0, look at the groups, run
`ttest_ind`, translate the p-value into a plain-English recommendation with the actual effect
size named).

(a) Does **infill pattern** (`grid` vs. `honeycomb`) affect **elongation (%)**? State H0 in
plain words, print each group's `n`, mean, and std, run `stats.ttest_ind`, and write one
sentence a shop manager could act on.

(b) Does **material** (`abs` vs. `pla`) affect **surface roughness** (`roughness (microns)`)?
Same four steps: state H0, print group stats, run `stats.ttest_ind`, write the shop-manager
sentence.

(c) Lecture 28's `material` vs. `tension_strength` result was p ≈ 0.041 -- just *under* the
0.05 line, called "real but borderline." In 2-3 sentences: is part (b)'s p-value above or below
0.05, and is it also close to the line? If a p-value just above 0.05 and a p-value just below
it aren't two different physical realities (Lecture 28's phrase), what would you actually tell
a shop manager to do about material choice and roughness -- trust the test's verdict outright,
or ask for more prints first? Why?

## Data file for Problem 4

## Dataset: Ceramic Fracture Strength -- Weibull Statistics (synthetic, calibrated)

Ceramics fail from their worst flaw, not their average flaw -- brittle strength is
**weakest-link, flaw-governed**, so it follows a Weibull distribution rather than a
normal one. The Weibull modulus `m` measures the *spread*: a low `m` means strength
is unpredictable (wide scatter, a fat low tail), a high `m` means strength is
reliable (tight scatter). Synthetic data, generated from literature Weibull moduli
for alumina, silicon carbide, and soda-lime glass (synthesis parameters documented
in this module's source header and docs/2026-07-22-new-dataset-verification.md).

In [5]:
import os
import pandas as pd

_file = 'weibull_ceramic_strength.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

print(f'{len(data)} specimens across {data["material"].nunique()} materials')
print(data.groupby(['material', 'batch'])['fracture_strength_mpa'].agg(['count', 'mean', 'std']))
data.head()

110 specimens across 3 materials
                count        mean        std
material batch                              
alumina  A         30  319.993000  40.749045
         B         30  298.423000  36.789527
glass    A         20   92.807500  22.798632
sic      A         15  378.853333  53.879244
         B         15  359.016000  51.015387


,material,batch,fracture_strength_mpa
0,alumina,A,334.67
1,alumina,A,363.23
2,alumina,A,366.20
3,alumina,A,331.22
4,alumina,A,326.89


# Problem 4 (25 pts): ceramics don't average, they weakest-link

Lecture 27 mentioned this data in passing: brittle fracture strength is **weakest-link**
physics (the specimen fails at its worst flaw), so it follows a Weibull distribution, not a
normal one -- a low Weibull modulus `m` means strength is unpredictable, a high `m` means it's
reliable. Lecture 28 named that this dataset has "some batch pairs [that] show a real strength
shift, at least one pair [that] is statistically indistinguishable" without saying which is
which. Your job below is to find out by running the numbers -- not to guess.

The include above already printed each material/batch's count, mean, and std -- look at that
table before starting.

(a) Run `stats.ttest_ind` comparing **alumina** batch A against batch B (filter with
`data.material == 'alumina'` and `data.batch == 'A'`/`'B'`, on the `fracture_strength_mpa`
column). Report the t-statistic and p-value, and a one-sentence plain-English conclusion.

(b) Do the same for **silicon carbide** (`data.material == 'sic'`), batch A vs. batch B.
Report t, p, and a one-sentence conclusion.

(c) A t-test compares means. It says nothing about *spread* -- and spread is exactly what the
Weibull modulus measures. Use this provided fitting function (the same recipe Lecture 27's
ceramics aside referenced, given here so you don't have to derive the log-log regression
yourself):

```python
from scipy import stats

def weibull_modulus_fit(strengths):
    "Fit the Weibull modulus m via linregress on the log-log-transformed sorted data."
    s = np.sort(np.asarray(strengths))
    n = len(s)
    F = (np.arange(1, n + 1) - 0.5) / n          # median-rank probability estimator
    x, yv = np.log(s), np.log(np.log(1 / (1 - F)))
    res = stats.linregress(x, yv)
    return res.slope, res.rvalue**2
```

Apply it to all five groups (`alumina`/A, `alumina`/B, `sic`/A, `sic`/B, `glass`). Print each
group's fitted `m` and R². Which group has the **lowest** `m` (least predictable strength)?
Which has the **highest** (most reliable)?

(d) In 3-4 sentences: for alumina, batch B's *mean* strength differs from batch A's (part a).
Does batch B's *modulus* (part c) also differ meaningfully from batch A's, or are the two
moduli close? What does that tell you about what actually changed between the batches -- a
shift in typical strength, a change in how reliable/predictable the strength is, or both?
(Ground your answer in the actual numbers from (a) and (c), not just the general idea that
"lower m is worse.")

# Wrap-up

Today closed out Module 2's toolkit: a category target gets kNN and accuracy instead of a
fitted line and R² (Problem 1); a suspicious point gets a quantitative rule and a judgment call,
never a silent deletion (Problem 2); a suspicious *difference between groups* gets a null
hypothesis and a p-value, translated into a sentence someone could act on (Problem 3); and a
material that fails from its worst flaw, not its average one, needs a distribution-shaped tool
(the Weibull modulus) alongside the mean-comparison tools you already had (Problem 4). None of
these are new universes of ideas -- they're the same fit-evaluate-interpret discipline from
Lectures 19-20, applied to a label, a flagged point, a comparison, and a distribution's shape,
in turn.